In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 100.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install -U transformers datasets accelerate scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
   

In [3]:
!pip install transformers datasets accelerate scikit-learn pandas

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving testdata.manual.2009.06.14.csv to testdata.manual.2009.06.14.csv
Saving train_14k_balanced .csv to train_14k_balanced .csv
Saving train_90k.csv to train_90k.csv


In [9]:
import re
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
# =========================
# 1. قراءة الداتا
# =========================
df = pd.read_csv("train_14k_balanced .csv")

print("Columns:", df.columns.tolist())
print("Rows:", len(df))

# نأخذ فقط الأعمدة المهمة
df = df[["text", "target"]].dropna()

# =========================
# 2. تنظيف النصوص
# تنظيف خفيف يناسب تويتر
# =========================
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)   # حذف الروابط
    text = re.sub(r"@\w+", "@user", text)        # توحيد المنشن
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)

# =========================
# 3. تحويل الليبلز
# Sentiment140: 0 = negative, 4 = positive
# نحولها إلى 0 و 1
# =========================
df["labels"] = df["target"].replace({0: 0, 4: 1}).astype(int)

print("\nLabel distribution:")
print(df["labels"].value_counts())

# =========================
# 4. تقسيم Train / Test
# =========================
train_df, test_df = train_test_split(
    df[["text", "labels"]],
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTrain size:", len(train_df))
print("Test size:", len(test_df))

# =========================
# 5. تحويل إلى HuggingFace Dataset
# =========================
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# =========================
# 6. تحميل مودل تويتر
# =========================
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# مهم:
# المودل الأصلي 3 فئات (neg/neu/pos)
# لكن نحن نريد فئتين فقط
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# =========================
# 7. دالة التقييم
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# =========================
# 8. إعدادات التدريب
# =========================
training_args = TrainingArguments(
    output_dir="./twitter_roberta_sentiment_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    disable_tqdm=True
)

# =========================
# 9. Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# =========================
# 10. التدريب
# =========================
trainer.train()
print("Training finished.")

# =========================
# 11. التقييم
# =========================
metrics = trainer.evaluate(eval_dataset=test_dataset)
print("\nFinal evaluation:")
print(metrics)

# =========================
# 12. حفظ المودل
# =========================
trainer.save_model("./twitter_roberta_sentiment_model")
tokenizer.save_pretrained("./twitter_roberta_sentiment_model")

print("\nModel saved in: twitter_roberta_sentiment_model")

import shutil

shutil.copytree(
    "/content/twitter_roberta_sentiment_model",
    "/content/drive/MyDrive/CoCare/models/twitter_roberta_sentiment_model",
    dirs_exist_ok=True
)

print("Model copied to Drive ")


Columns: ['target', 'id', 'date', 'flag', 'user', 'text']
Rows: 14000

Label distribution:
labels
0    7000
1    7000
Name: count, dtype: int64

Train size: 11200
Test size: 2800


Map:   0%|          | 0/11200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

{'train_runtime': '241.7', 'train_samples_per_second': '92.66', 'train_steps_per_second': '5.791', 'train_loss': '0.3336', 'epoch': '2'}


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training finished.
{'eval_loss': '0.3652', 'eval_accuracy': '0.8457', 'eval_precision': '0.8775', 'eval_recall': '0.8036', 'eval_f1': '0.8389', 'eval_runtime': '2.784', 'eval_samples_per_second': '1006', 'eval_steps_per_second': '62.85', 'epoch': '2'}

Final evaluation:
{'eval_loss': 0.3652278780937195, 'eval_accuracy': 0.8457142857142858, 'eval_precision': 0.8775351014040562, 'eval_recall': 0.8035714285714286, 'eval_f1': 0.8389261744966443, 'eval_runtime': 2.7845, 'eval_samples_per_second': 1005.553, 'eval_steps_per_second': 62.847, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved in: twitter_roberta_sentiment_model
Model copied to Drive 


In [10]:
!zip -r twitter_roberta_sentiment_model.zip twitter_roberta_sentiment_model
from google.colab import files
files.download("twitter_roberta_sentiment_model.zip")


  adding: twitter_roberta_sentiment_model/ (stored 0%)
  adding: twitter_roberta_sentiment_model/tokenizer_config.json (deflated 50%)
  adding: twitter_roberta_sentiment_model/training_args.bin (deflated 53%)
  adding: twitter_roberta_sentiment_model/tokenizer.json (deflated 82%)
  adding: twitter_roberta_sentiment_model/config.json (deflated 52%)
  adding: twitter_roberta_sentiment_model/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "./twitter_roberta_sentiment_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

label_map = {0: "negative", 1: "positive"}

texts = [
    "I love this service so much!",
    "This internet is terrible.",
    "Support team was amazing.",
    "I hate this connection."
]

for text in texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    print("Text:", text)
    print("Prediction:", label_map[pred])
    print("-" * 40)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: I love this service so much!
Prediction: positive
----------------------------------------
Text: This internet is terrible.
Prediction: negative
----------------------------------------
Text: Support team was amazing.
Prediction: positive
----------------------------------------
Text: I hate this connection.
Prediction: negative
----------------------------------------


In [19]:
from sklearn.metrics import accuracy_score
import numpy as np

# ===== Testing Accuracy =====
test_output = trainer.predict(test_dataset)

y_test_pred = np.argmax(test_output.predictions, axis=1)
y_test_true = test_output.label_ids

test_acc = accuracy_score(y_test_true, y_test_pred)

print("=" * 40)
print("TEST ACCURACY")
print("=" * 40)
print("Testing Accuracy:", round(test_acc, 4))

TEST ACCURACY
Testing Accuracy: 0.8457
